# Разметка дыхания эксперимента 2

**Статус:** активный производитель кандидатов дыхательной разметки. Результат
становится проверенным артефактом только после ручного контроля качества.

Ноутбук выделен из исторического 08_Скетч_разметки.ipynb. Исходная смешанная
реализация сохранена в archive/legacy/11.90_Скетч_механических_точек.ipynb,
а точное происхождение ячеек — в MIGRATION_MANIFEST_2026-08-26.json.


## Входы, допущения и выход

Входом служат внешние CSV эксперимента 2 с колонками времени и базового
импеданса боковой сборки. Пути и идентификаторы испытуемых задаются только во
внешнем локальном JSON по схеме config/exp02_paths.example.json.

Алгоритм ищет два продолжительных участка с малой локальной вариабельностью и
назначает им кандидаты задержек вдоха и выдоха по уровню базового импеданса.
Остальные интервалы формируются из порядка этих участков. Такая процедура
является эвристикой, а соответствие режимам — допущением до ручной проверки.
Контактный дефект, дрейф или слабое дыхание могут давать похожие участки.

Каждый JSON получает хеш входного файла, версию алгоритма и статус
pending_manual_review. Последующие расчёты должны принимать только артефакты
со статусом accepted.


In [ ]:
# Импорты и внешняя конфигурация
import hashlib
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CONFIG_ENV = "KALMYKOV_EXP02_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp02_paths.example.json"
    )

CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
SUBJECTS = {item["subject_id"]: item for item in CONFIG["subjects"]}

OUT_DIR = DERIVED_ROOT / "exp02" / "annotations" / "breathing"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "TIME_s"
SEG_COL = "BASE_2_Ω"
ALGORITHM_VERSION = "exp02-breathing-heuristic-v1"
DUPLICATE_SIZE_MM = {100: 90}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def sampling_frequency(df):
    time = df[TIME_COL].to_numpy(dtype=float)
    delta = np.diff(time)
    if len(delta) == 0 or np.any(~np.isfinite(delta)) or np.any(delta <= 0):
        raise ValueError("TIME_s должен быть конечным и строго возрастающим")
    median_dt = float(np.median(delta))
    jitter_fraction = float(
        np.median(np.abs(delta - median_dt)) / median_dt
    )
    return 1.0 / median_dt, jitter_fraction

def iter_recordings():
    for subject_id, info in SUBJECTS.items():
        directory = DATA_ROOT / info["data_subdir"]
        suffix = info["filename_suffix"]
        pattern = re.compile(rf"^(\d+){re.escape(suffix)}\.csv$")
        for path in sorted(directory.glob(f"*{suffix}.csv")):
            match = pattern.match(path.name)
            if not match:
                continue
            size_mm = int(match.group(1))
            if size_mm in DUPLICATE_SIZE_MM:
                continue
            yield subject_id, info, size_mm, path


In [ ]:
# Эвристическое выделение кандидатов дыхательных режимов
def segment_breathing(df, win_s=1.0, min_hold_s=4.0):
    time = df[TIME_COL].to_numpy(dtype=float)
    signal = df[SEG_COL].to_numpy(dtype=float)
    fs, _ = sampling_frequency(df)

    window = max(5, int(win_s * fs))
    rolling_std = (
        pd.Series(signal)
        .rolling(window, center=True, min_periods=max(3, window // 2))
        .std()
        .to_numpy()
    )
    threshold = max(0.06, float(np.nanpercentile(rolling_std, 20) * 2.0))
    quiet = rolling_std < threshold

    runs = []
    index = 0
    while index < len(quiet):
        if not quiet[index]:
            index += 1
            continue
        stop = index
        while stop < len(quiet) and quiet[stop]:
            stop += 1
        if stop > index and time[stop - 1] - time[index] >= min_hold_s:
            runs.append((index, stop))
        index = stop

    runs.sort(key=lambda item: -(time[item[1] - 1] - time[item[0]]))
    holds = sorted(runs[:2], key=lambda item: item[0])
    if len(holds) < 2:
        return None

    first, second = holds
    level_first = float(np.median(signal[first[0]:first[1]]))
    level_second = float(np.median(signal[second[0]:second[1]]))
    inhale, exhale = (
        (first, second) if level_first > level_second else (second, first)
    )

    def interval(segment):
        return [
            float(time[segment[0]]),
            float(time[min(segment[1], len(time) - 1)]),
        ]

    return {
        "candidate_modes": {
            "спокойное": [float(time[0]), float(time[inhale[0]])],
            "задержка_вдох": interval(inhale),
            "глубокое": [float(time[inhale[1]]), float(time[exhale[0]])],
            "задержка_выдох": interval(exhale),
        },
        "hold_levels_ohm": {
            "вдох": max(level_first, level_second),
            "выдох": min(level_first, level_second),
        },
        "heuristic": {
            "window_s": win_s,
            "min_hold_s": min_hold_s,
            "quiet_threshold_channel_units": threshold,
        },
    }


In [ ]:
# Построение отдельных дыхательных sidecar-файлов
records = []
for subject_id, info, size_mm, source_path in iter_recordings():
    frame = pd.read_csv(source_path, encoding="utf-8")
    fs_hz, jitter_fraction = sampling_frequency(frame)
    candidate = segment_breathing(frame)
    if candidate is None:
        print("Не найдены две задержки:", source_path.name)
        continue

    input_sha256 = sha256_file(source_path)
    record_id = input_sha256[:16]
    relative_path = source_path.relative_to(DATA_ROOT).as_posix()
    output = {
        "schema_version": 1,
        "annotation_type": "breathing",
        "algorithm_version": ALGORITHM_VERSION,
        "record_id": record_id,
        "subject_id": subject_id,
        "size_mm": size_mm,
        "input": {
            "relative_path": relative_path,
            "sha256": input_sha256,
            "sampling_frequency_hz": fs_hz,
            "sampling_frequency_source": "median_diff_TIME_s",
            "relative_time_step_jitter": jitter_fraction,
        },
        **candidate,
        "assumptions": [
            "two_longest_quiet_segments_are_breath_holds",
            "higher_BASE_2_level_corresponds_to_inhale",
            "recordings_of_different_sizes_are_comparable",
        ],
        "qc": {
            "status": "pending_manual_review",
            "reviewer": None,
            "reviewed_at": None,
            "notes": None,
        },
    }
    output_path = OUT_DIR / f"{record_id}.json"
    output_path.write_text(
        json.dumps(output, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    records.append(output)
    print(record_id, size_mm, output["qc"]["status"])

print("Создано кандидатов:", len(records))
print("Каталог:", OUT_DIR)


## Ручной контроль качества

Для каждого файла необходимо проверить форму BASE_2_Ω, соответствие
кандидатных интервалов командам протокола и отсутствие контактных артефактов.
Статус accepted устанавливается только человеком после просмотра конкретной
записи. Автоматический график ниже не изменяет sidecar и не считается
валидацией.


In [ ]:
# Контрольный график выбранного кандидата без автоматического принятия
CHECK_RECORD_ID = None

if CHECK_RECORD_ID is None:
    print("Задайте CHECK_RECORD_ID после построения кандидатов.")
else:
    sidecar_path = OUT_DIR / f"{CHECK_RECORD_ID}.json"
    annotation = json.loads(sidecar_path.read_text(encoding="utf-8"))
    source_path = DATA_ROOT / annotation["input"]["relative_path"]
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("Хеш исходного CSV изменился после разметки")

    frame = pd.read_csv(source_path, encoding="utf-8")
    time = frame[TIME_COL].to_numpy(dtype=float)
    fig, axis = plt.subplots(figsize=(14, 4))
    axis.plot(time, frame[SEG_COL], color="navy", linewidth=0.7)
    for name, (start, stop) in annotation["candidate_modes"].items():
        axis.axvspan(start, stop, alpha=0.2, label=name)
    axis.set_xlabel("Время, с")
    axis.set_ylabel("BASE_2, Ом")
    axis.set_title(
        f"{CHECK_RECORD_ID}: кандидат дыхательной разметки; "
        f"QC={annotation['qc']['status']}"
    )
    axis.legend()
    plt.tight_layout()
    plt.show()


## Выход и зависимые этапы

Выход находится во внешнем каталоге
derived/exp02/annotations/breathing/. В Git допускаются только схема и
обезличенный пример. 11.02_Разметка_ЭКГ_эксперимента_2.ipynb связывает
ЭКГ-разметку с той же записью по record_id и хешу входного CSV.
